# Day 39 — Solutions

## Decision Trees

**Boston University · Data Science Program**

---

15 questions in four parts.

| part | topic | questions |
|---|---|---|
| **A** | Reading a tree | 4 |
| **B** | How a split is chosen | 4 |
| **C** | Overfitting and pruning | 4 |
| **D** | Judgement | 3 |

Every answer below was produced by running the code, not by hand.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore")

from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     cross_validate, GridSearchCV)

CV = StratifiedKFold(5, shuffle=True, random_state=0)

# ---- the same 900 scholarship decisions as the lesson -------------------
rng = np.random.default_rng(3900)
N = 900
marks         = rng.normal(70, 14, N).clip(35, 100).round(0)
family_income = rng.normal(380000, 210000, N).clip(40000, 1500000).round(0)
attendance    = rng.normal(82, 12, N).clip(40, 100).round(0)
sport         = rng.binomial(1, 0.25, N).astype(float)
rule = (((marks >= 75) & (attendance >= 80)) | (marks >= 88)
        | ((sport == 1) & (marks >= 62) & (attendance >= 75)))
rule = rule & (family_income <= 700000)
flip = rng.uniform(0, 1, N) < 0.08
awarded = np.where(flip, 1 - rule.astype(int), rule.astype(int))

sch = pd.DataFrame({"marks": marks, "family_income": family_income,
                    "attendance": attendance, "sport": sport, "awarded": awarded})
X = sch[["marks", "family_income", "attendance", "sport"]]
y = sch["awarded"]

def gini(labels):
    labels = np.asarray(labels)
    if len(labels) == 0:
        return 0.0
    p = labels.mean()
    return 1 - p**2 - (1 - p)**2

print("setup ok —", len(sch), "rows, baseline", round(1 - y.mean(), 4))

---
# Part A — Reading a tree

### A1
Fit a `DecisionTreeClassifier` with `max_depth=2` and `random_state=0` on all of `X, y`,
and print it with `export_text`.

1. What question is at the root?
2. How many leaves does it have?
3. What fraction of the 900 students land in the biggest leaf?
4. Look at the two leaves under `marks <= 74.5`. What is odd about them, and what does
   that tell you about the split above them?

In [ ]:
# your code here

In [ ]:
t2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X, y)
print(export_text(t2, feature_names=list(X.columns), decimals=1))

leaf_of = t2.apply(X)
counts = pd.Series(leaf_of).value_counts()
print("leaves:", t2.get_n_leaves())
print("biggest leaf holds", counts.iloc[0], "students =",
      round(counts.iloc[0] / len(X), 4), "of the data")
print()
for lid in sorted(set(leaf_of)):
    m = leaf_of == lid
    print(f"  leaf {lid}: n={m.sum():>3}  awarded={int(y[m].sum()):>3} "
          f"({y[m].mean():.3f})  -> predicts {int(y[m].mean() > 0.5)}")

**A1 answer**

1. The root question is **`marks <= 74.5`**.
2. **4 leaves.**
3. The biggest leaf holds **452** students = **0.5022**, just over half the data.
4. Both leaves under `marks <= 74.5` **predict the same class (0)**. The `sport <= 0.5`
   split below it did not change any decision — it only separated a group that is 83%
   "not awarded" from one that is 58% "not awarded". It sharpened the *confidence*
   without changing the *answer*.

   That is normal and not a bug. A split is chosen because it lowers impurity, not
   because it flips a prediction. Give the tree one more level of depth and that
   58% group does split into a real decision.

### A2
Here is a new applicant:

| marks | family_income | attendance | sport |
|---|---|---|---|
| 66 | 180000 | 91 | 1 |

Fit a **depth-3** tree. Before running `predict`, walk this student down the tree by hand
using the `export_text` output and write down which leaf you expect.

Then check with `predict` and `predict_proba`. Do they match?

In [ ]:
# your code here

In [ ]:
t3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, y)
print(export_text(t3, feature_names=list(X.columns), decimals=1))

student = pd.DataFrame([{"marks": 66.0, "family_income": 180000.0,
                         "attendance": 91.0, "sport": 1.0}])
print("by hand:")
print("  marks <= 74.5 ?  66 <= 74.5   -> YES, go left")
print("  sport <= 0.5  ?  1  >  0.5    -> NO,  go right")
print("  marks <= 61.5 ?  66 >  61.5   -> NO,  go right")
print()
print("predict      ", t3.predict(student)[0])
print("predict_proba", t3.predict_proba(student)[0].round(4))

**A2 answer**

The path is: `marks <= 74.5` **yes** → `sport <= 0.5` **no** → `marks <= 61.5` **no**.

That lands in the leaf holding 23 not-awarded and 52 awarded students.

- `predict` returns **1** (awarded)
- `predict_proba` returns **[0.3067, 0.6933]**

And 52 / 75 = 0.6933, so the hand trace and sklearn agree exactly.

Notice *why* this student is awarded despite middling marks: they play a sport and their
attendance is high, which is one of the committee's three routes to an award. A model
that only looked at marks would have denied them.

### A3
Confirm that `predict_proba` really is just a count. Use `tree.apply()` to find which
leaf the A2 student landed in, pull out every *training* student in that same leaf, and
compute the fraction who were awarded yourself.

In [ ]:
# your code here

In [ ]:
leaf_id = t3.apply(student)[0]
same_leaf = t3.apply(X) == leaf_id

print("leaf id:", leaf_id)
print("training students there:", same_leaf.sum())
print(y[same_leaf].value_counts().rename({0: "not awarded", 1: "awarded"}).to_string())
print()
print("fraction awarded by hand :", round(y[same_leaf].mean(), 4))
print("predict_proba[1]         :", round(t3.predict_proba(student)[0][1], 4))

**A3 answer**

Leaf holds **75** training students: 23 not awarded, 52 awarded.
By hand, 52/75 = **0.6933**, which is exactly `predict_proba[1]`.

This is worth internalising: a tree's "probability" is **the fraction of training rows
in that leaf** and nothing more. So the probability from a leaf containing 4 students is
not a probability in any useful sense — it is `0.25`, `0.50` or `0.75` and it will move
if you change the data slightly. That is one more reason to forbid tiny leaves.

### A4
Multiply the `attendance` column by 100 and refit:

- a `DecisionTreeClassifier(max_depth=4, random_state=0)`
- a `KNeighborsClassifier(15)` with **no** scaler

Report the 5-fold CV accuracy of each, before and after. Explain the difference in one
sentence each.

In [ ]:
# your code here

In [ ]:
X_big = X.copy()
X_big["attendance"] = X_big["attendance"] * 100

rows = []
for name, data in [("as it comes", X), ("attendance x 100", X_big)]:
    rows.append({
        "version": name,
        "tree":  round(cross_val_score(DecisionTreeClassifier(max_depth=4, random_state=0),
                                       data, y, cv=CV).mean(), 4),
        "k-NN":  round(cross_val_score(KNeighborsClassifier(15), data, y, cv=CV).mean(), 4),
    })
print(pd.DataFrame(rows).to_string(index=False))

**A4 answer**

| version | tree | k-NN |
|---|---|---|
| as it comes | 0.8967 | 0.6711 |
| attendance × 100 | **0.8967** | **0.6589** |

- **The tree is unchanged, exactly.** It only asks "is attendance above or below some
  number?", and when the column is multiplied by 100 the best threshold is multiplied
  by 100 too — the same students go the same way.
- **k-NN moves**, because it adds up squared differences across columns, so blowing up
  one column's units makes that column dominate the distance and drowns out the others.

Note k-NN scored only 0.6711 even *before* the change: `family_income` is already in
hundreds of thousands, so it was already dominating. Unscaled k-NN was barely above the
0.6678 baseline the whole time.

---
# Part B — How a split is chosen

### B1
Take only the students who play a sport (`sport == 1`).

How many are there, what fraction were awarded, and what is the Gini impurity of that
group? Is this group easier or harder to predict than the full 900?

In [ ]:
# your code here

In [ ]:
grp = y[X["sport"] == 1]
print("students who play a sport:", len(grp))
print("awarded:", int(grp.sum()), f"({grp.mean():.4f})")
print("gini   :", round(gini(grp), 4))
print()
print("the full 900 for comparison: gini", round(gini(y), 4))

**B1 answer**

- **228** students play a sport
- **117** of them were awarded, a fraction of **0.5132**
- Gini impurity **0.4997**

The full dataset has Gini **0.4437**, so the sport group is **harder**, not easier —
0.4997 is almost exactly the theoretical maximum of 0.5.

This is a useful surprise. "Plays a sport" sounds like strong evidence, and it *is* part
of the committee's rule — but only in combination with marks and attendance. On its own
it splits the group into a near-perfect coin flip. **A column can be genuinely important
and still be useless as a single question**, which is exactly why the tree keeps asking
more questions underneath.

### B2
Score the split `attendance <= 85` the way the tree does: Gini on each side, weighted by
group size, then the drop from the parent's 0.4437.

Compare it with the root split `marks <= 74.5`. By how many times is the root better?

In [ ]:
# your code here

In [ ]:
def score_split(col, threshold):
    m = X[col] <= threshold
    left, right = y[m], y[~m]
    weighted = (len(left) * gini(left) + len(right) * gini(right)) / len(y)
    return weighted, len(left), gini(left), len(right), gini(right)

for col, t in [("attendance", 85.0), ("marks", 74.5)]:
    w, nl, gl, nr, gr = score_split(col, t)
    print(f"{col} <= {t}")
    print(f"   LEFT  n={nl:>3} gini {gl:.4f}   RIGHT n={nr:>3} gini {gr:.4f}")
    print(f"   weighted {w:.4f}   drop {gini(y) - w:.4f}")

d_att = gini(y) - score_split("attendance", 85.0)[0]
d_mrk = gini(y) - score_split("marks", 74.5)[0]
print(f"\nthe root is {d_mrk / d_att:.1f} times better")

**B2 answer**

| question | left | right | weighted Gini | drop |
|---|---|---|---|---|
| `attendance <= 85` | n=562, gini 0.4088 | n=338, gini 0.4832 | 0.4367 | **0.0070** |
| `marks <= 74.5` | n=592, gini 0.2763 | n=308, gini 0.4534 | 0.3369 | **0.1068** |

The root split is **15.3 times better**.

`attendance <= 85` barely helps: both sides come out nearly as mixed as the parent
(0.4088 and 0.4832 against the parent's 0.4437). It is not that attendance is irrelevant
— it appears twice in the real tree — it is that **85 is the wrong place to cut it**,
which B3 shows.

### B3
Find the *best* threshold on `attendance` by scanning every midpoint between consecutive
distinct values, exactly as the tree does.

How many candidates are there, which wins, and how does its impurity drop compare with
the root's 0.1068?

In [ ]:
# your code here

In [ ]:
vals = np.unique(X["attendance"])
mids = (vals[:-1] + vals[1:]) / 2
scores = np.array([score_split("attendance", t)[0] for t in mids])

best_i = int(scores.argmin())
print("candidate thresholds on attendance:", len(mids))
print(f"best: attendance <= {mids[best_i]}  weighted gini {scores[best_i]:.4f}"
      f"  drop {gini(y) - scores[best_i]:.4f}")
print(f"the root (marks <= 74.5) drop was 0.1068")

plt.figure(figsize=(9, 4))
plt.plot(mids, scores, "o-", ms=4)
plt.plot(mids[best_i], scores[best_i], "o", ms=12, color="#1F9E86")
plt.axhline(gini(y), ls="--", color="#E05C45", label="before any split")
plt.xlabel("threshold on attendance"); plt.ylabel("weighted Gini after")
plt.legend(); plt.title("scanning one column")
plt.tight_layout(); plt.show()

**B3 answer**

- **50** candidate thresholds
- The best is **`attendance <= 75.5`**, weighted Gini **0.4208**, a drop of **0.0229**

So the best possible attendance question is still only about **a fifth** as good as
`marks <= 74.5` (0.0229 against 0.1068). The tree is right to put marks at the root.

But compare 0.0229 with B2's 0.0070: **choosing the threshold well made the attendance
question three times more useful.** This is why the tree scans every midpoint instead of
picking round numbers — and why you should never hand-pick thresholds for it.

### B4
Why does the tree weight each side by how many students are in it, instead of just
averaging the two Gini values?

Construct a small example to show what goes wrong. Imagine a group of 100 students,
50 awarded and 50 not, and compare these two candidate splits:

- **Split A** — left: 3 students, all awarded. right: the other 97.
- **Split B** — left: 50 students, 45 awarded. right: 50 students, 5 awarded.

Compute both the **unweighted average** and the **weighted average** of the two Gini
values for each split, and say which split each method prefers.

In [ ]:
# your code here

In [ ]:
def g_from(n_awarded, n_total):
    if n_total == 0:
        return 0.0
    p = n_awarded / n_total
    return 1 - p**2 - (1 - p)**2

splits = {
    "A  (3 pure | 97 mixed)": ((3, 3), (47, 97)),
    "B  (45/50   | 5/50)":    ((45, 50), (5, 50)),
}
print(f"parent: 50 of 100 awarded, gini {g_from(50, 100):.4f}\n")
for name, (L, R) in splits.items():
    gl, gr = g_from(*L), g_from(*R)
    unweighted = (gl + gr) / 2
    weighted   = (L[1] * gl + R[1] * gr) / 100
    print(f"{name}")
    print(f"   left gini {gl:.4f} (n={L[1]})   right gini {gr:.4f} (n={R[1]})")
    print(f"   unweighted average {unweighted:.4f}      weighted average {weighted:.4f}")

**B4 answer**

| split | left gini | right gini | unweighted avg | weighted avg |
|---|---|---|---|---|
| A (3 pure \| 97 mixed) | 0.0000 (n=3) | 0.4995 (n=97) | **0.2498** | 0.4845 |
| B (45/50 \| 5/50) | 0.1800 (n=50) | 0.1800 (n=50) | 0.1800 | **0.1800** |

Here B happens to win on both measures — but look at *how much*: unweighted, split A
scores 0.2498 and looks respectable, because the perfectly pure 3-student group gets
**the same vote as the 97-student group it left behind**. Weighted, A scores 0.4845,
barely better than the parent's 0.5000, which is the truth: it sorted out 3 students and
achieved essentially nothing.

Make split A's pure group a little larger relative to B's advantage and unweighted
averaging will actively pick the wrong one. Weighting asks the right question — **"how
much total impurity did we remove across all the students?"** — rather than "how pretty
are the two groups on average?".

This is also why `min_samples_leaf` exists: even with weighting, the tree is happy to
carve off tiny pure groups once it runs out of better ideas.

---
# Part C — Overfitting and pruning

### C1
Build a table of `max_depth` in `[1, 2, 3, 4, 5, 6, 8, None]` showing, for each:
number of leaves, mean **training** accuracy, mean **cross-validated** accuracy, and the gap.

Use `cross_validate(..., return_train_score=True)` with `CV`.

Which depth is best? What is the gap at `None`, and what is it at the best depth?

In [ ]:
# your code here

In [ ]:
rows = []
for d in [1, 2, 3, 4, 5, 6, 8, None]:
    r = cross_validate(DecisionTreeClassifier(max_depth=d, random_state=0),
                       X, y, cv=CV, return_train_score=True)
    leaves = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X, y).get_n_leaves()
    rows.append({"max_depth": "none" if d is None else d, "leaves": leaves,
                 "train": round(r["train_score"].mean(), 4),
                 "CV":    round(r["test_score"].mean(), 4),
                 "gap":   round(r["train_score"].mean() - r["test_score"].mean(), 4)})
print(pd.DataFrame(rows).to_string(index=False))

**C1 answer**

| max_depth | leaves | train | CV | gap |
|---|---|---|---|---|
| 1 | 2 | 0.7722 | 0.7722 | +0.0000 |
| 2 | 4 | 0.8156 | 0.8078 | +0.0078 |
| 3 | 8 | 0.8869 | 0.8811 | +0.0058 |
| 4 | 15 | 0.9092 | 0.8967 | +0.0125 |
| **5** | 26 | 0.9239 | **0.9078** | +0.0161 |
| 6 | 44 | 0.9358 | 0.8956 | +0.0403 |
| 8 | 75 | 0.9622 | 0.8844 | +0.0778 |
| none | 137 | 1.0000 | 0.8478 | **+0.1522** |

**Depth 5 is best**, at CV 0.9078 with a gap of only +0.0161.

At `None` the gap is **+0.1522** — training accuracy is a perfect 1.0000 while the
honest score has fallen to 0.8478, *worse than depth 3*. Notice the training column rises
all the way to the bottom of the table. If you chose by it you would pick the worst model.

### C2
Now hold depth unlimited and vary `min_samples_leaf` over `[1, 2, 5, 10, 20, 50, 100]`.

Which value gives the best CV score? Which gives the smallest gap? Are they the same
value, and what should you do when they disagree?

In [ ]:
# your code here

In [ ]:
rows = []
for m in [1, 2, 5, 10, 20, 50, 100]:
    r = cross_validate(DecisionTreeClassifier(min_samples_leaf=m, random_state=0),
                       X, y, cv=CV, return_train_score=True)
    leaves = DecisionTreeClassifier(min_samples_leaf=m, random_state=0).fit(X, y).get_n_leaves()
    rows.append({"min_samples_leaf": m, "leaves": leaves,
                 "train": round(r["train_score"].mean(), 4),
                 "CV":    round(r["test_score"].mean(), 4),
                 "gap":   round(r["train_score"].mean() - r["test_score"].mean(), 4)})
tbl = pd.DataFrame(rows)
print(tbl.to_string(index=False))
print()
print("best CV      :", tbl.loc[tbl.CV.idxmax(), "min_samples_leaf"])
print("smallest |gap|:", tbl.loc[tbl.gap.abs().idxmin(), "min_samples_leaf"])

**C2 answer**

| min_samples_leaf | leaves | train | CV | gap |
|---|---|---|---|---|
| 1 | 137 | 1.0000 | 0.8478 | +0.1522 |
| 2 | 113 | 0.9525 | 0.8611 | +0.0914 |
| 5 | 76 | 0.9219 | 0.8744 | +0.0475 |
| **10** | 50 | 0.9050 | **0.8956** | +0.0094 |
| 20 | 31 | 0.8828 | 0.8711 | +0.0117 |
| 50 | 13 | 0.8450 | 0.8478 | **−0.0028** |
| 100 | 7 | 0.8133 | 0.8122 | +0.0011 |

- Best **CV**: `min_samples_leaf = 10` (0.8956)
- Smallest gap in absolute size: `min_samples_leaf = 100` (**+0.0011**)

They disagree, and **you should follow the CV score, not the gap.** A small gap only
says training and new data agree — it says nothing about whether they agree on something
*good*. At 100 the gap is almost exactly zero and the CV score has fallen to 0.8122,
heading back towards the 0.6678 baseline: the model has become too simple to overfit,
which is not the same as being right.

Look at `min_samples_leaf = 50` too, where the gap is **negative** (−0.0028): the tree
scored *higher* on data it had never seen than on data it trained on. That is not a
miracle, it is noise — with 13 leaves the model is so blunt that the difference between
the two numbers is just which rows happened to fall in which fold. A negative gap is
another underfitting signal, not a bonus.

**A zero gap with a bad score is underfitting.** The gap is a diagnostic, never the target.

### C3
Use `GridSearchCV` to search `max_depth` and `min_samples_leaf` together.

How many models does it fit in total? What does it pick, and what does it score?

In [ ]:
# your code here

In [ ]:
param_grid = {"max_depth": [2, 3, 4, 5, 6, 8, None],
              "min_samples_leaf": [1, 5, 10, 20, 50]}

grid = GridSearchCV(DecisionTreeClassifier(random_state=0), param_grid, cv=CV, n_jobs=-1)
grid.fit(X, y)

n_combos = len(param_grid["max_depth"]) * len(param_grid["min_samples_leaf"])
print(f"{n_combos} combinations x 5 folds = {n_combos * 5} trees fitted")
print()
print("best params:", grid.best_params_)
print("best CV    :", round(grid.best_score_, 4))
print()
print("top 5:")
res = pd.DataFrame(grid.cv_results_)
cols = ["param_max_depth", "param_min_samples_leaf", "mean_test_score"]
print(res.sort_values("mean_test_score", ascending=False)[cols].head().to_string(index=False))

**C3 answer**

7 depths × 5 leaf sizes = **35 combinations**, each fitted on 5 folds = **175 trees**.

It picks **`max_depth=5, min_samples_leaf=1`**, scoring **0.9078** — the same answer C1
found by varying depth alone, which is reassuring.

Note that `min_samples_leaf=1` wins here only *because* depth 5 is already doing the
pruning. The two knobs overlap: once the depth cap stops the tree at 26 leaves, there are
no 2-student leaves left for `min_samples_leaf` to forbid. Searching them jointly is
still the right move, because you cannot know that in advance.

### C4
**The trap.** Suppose you had chosen `max_depth` by whichever value gave the highest
**training** accuracy, instead of the highest cross-validated accuracy.

Which depth would you have picked, what would it actually score on new data, and how much
accuracy would that mistake have cost you?

In [ ]:
# your code here

In [ ]:
rows = []
for d in [1, 2, 3, 4, 5, 6, 8, None]:
    r = cross_validate(DecisionTreeClassifier(max_depth=d, random_state=0),
                       X, y, cv=CV, return_train_score=True)
    rows.append({"depth": "none" if d is None else d,
                 "train": r["train_score"].mean(), "CV": r["test_score"].mean()})
tbl = pd.DataFrame(rows)

by_train = tbl.loc[tbl.train.idxmax()]
by_cv    = tbl.loc[tbl.CV.idxmax()]

print(f"chosen by TRAIN score : depth {by_train.depth}"
      f"  (train {by_train.train:.4f})  -> real score {by_train.CV:.4f}")
print(f"chosen by CV score    : depth {by_cv.depth}"
      f"  (train {by_cv.train:.4f})  -> real score {by_cv.CV:.4f}")
print(f"\ncost of the mistake   : {by_cv.CV - by_train.CV:.4f}")

**C4 answer**

- Picked by **training** score: `max_depth=None`, training accuracy a flawless **1.0000**
  — and a real score of **0.8478**.
- Picked by **CV** score: `max_depth=5`, training accuracy 0.9239, real score **0.9078**.

The mistake costs **0.0600** — six students in every hundred, given away in exchange for
a number that felt perfect.

Worse than the lost accuracy is the lost *honesty*: a model reporting 1.0000 invites you
to stop checking. The whole purpose of a held-out score is to stay suspicious of your
own model, and training accuracy on a tree is the least trustworthy number in this
entire course, because an unlimited tree can always drive it to 1.0.

---
# Part D — Judgement

### D1
Day 39's headline was that the tree beat logistic regression by nearly 10 points. That
was a fact about **this data**, not about trees.

Here is a different problem: predicting whether a student passes an exam, from hours
studied and their previous score. This time the truth is a **smooth** relationship, not a
set of thresholds.

Run the cell below to build it, then compare: logistic regression, k-NN (k=15), a
depth-4 tree, and a tuned tree. Report the baseline too.

Which model wins? Is the winner clearly ahead?

In [ ]:
r2 = np.random.default_rng(777)
M = 900
hours      = r2.normal(12, 4, M).clip(0, 30).round(1)
prev_score = r2.normal(60, 15, M).clip(10, 100).round(0)

# a smooth logistic truth: no thresholds anywhere
z = -7.5 + 0.30 * hours + 0.075 * prev_score
passed = (r2.uniform(0, 1, M) < 1 / (1 + np.exp(-z))).astype(int)

exam = pd.DataFrame({"hours": hours, "prev_score": prev_score, "passed": passed})
XL, yL = exam[["hours", "prev_score"]], exam["passed"]
print(exam.head())
print("\npass rate:", round(yL.mean(), 4))

In [ ]:
# your code here

In [ ]:
contenders = {
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "k-NN (k=15)":         make_pipeline(StandardScaler(), KNeighborsClassifier(15)),
    "tree (depth 4)":      DecisionTreeClassifier(max_depth=4, random_state=0),
    "tree (tuned)":        GridSearchCV(DecisionTreeClassifier(random_state=0),
                               {"max_depth": [2, 3, 4, 5, 6, None],
                                "min_samples_leaf": [1, 5, 10, 20]}, cv=CV),
}
print(f"  {'baseline (majority)':<22}{max(yL.mean(), 1 - yL.mean()):.4f}")
for name, model in contenders.items():
    print(f"  {name:<22}{cross_val_score(model, XL, yL, cv=CV).mean():.4f}")

**D1 answer**

| model | CV accuracy |
|---|---|
| baseline (majority) | 0.5911 |
| logistic regression | 0.7533 |
| k-NN (k=15) | **0.7589** |
| tree (depth 4) | 0.7322 |
| tree (tuned) | 0.7500 |

**The tree comes last**, and even after tuning it does not catch logistic regression.

But answer the second question carefully: k-NN tops the table at 0.7589 against
logistic's 0.7533 — a difference of **0.0056**, which is far smaller than the spread
between folds. **That gap is not evidence that k-NN is better here.** The honest
statement is that logistic regression and k-NN are tied, and the tree is behind both.

Why the tree loses: the truth is a smooth diagonal, and a tree can only cut horizontally
and vertically, so it approximates the diagonal with a staircase and gets the corners
wrong. On Day 39's scholarship data the truth *was* made of corners and the tree won by
10 points. Same model, opposite result.

> **The lesson:** there is no best algorithm. Pick the one whose shape matches the truth,
> and use cross-validation — not yesterday's result — to find out which that is.

### D2
Fit a `max_depth=5` tree on `X, y` and print `feature_importances_`.

Then add a column `marks_again` that is an exact copy of `marks`, refit, and print the
importances again.

What happens to the importance of `marks`? What does that tell you about reading
feature importances when your columns are correlated?

In [ ]:
# your code here

In [ ]:
t5 = DecisionTreeClassifier(max_depth=5, random_state=0).fit(X, y)
before = pd.Series(t5.feature_importances_, index=X.columns)
print("original columns:")
print(before.round(4).to_string())

X_dup = X.copy()
X_dup["marks_again"] = X_dup["marks"]
t5d = DecisionTreeClassifier(max_depth=5, random_state=0).fit(X_dup, y)
after = pd.Series(t5d.feature_importances_, index=X_dup.columns)
print("\nwith an exact duplicate of marks:")
print(after.round(4).to_string())

print(f"\nmarks alone        : {before['marks']:.4f}")
print(f"marks + marks_again: {after['marks'] + after['marks_again']:.4f}")

**D2 answer**

| column | original | with duplicate |
|---|---|---|
| marks | **0.5306** | **0.0906** |
| family_income | 0.1382 | 0.1382 |
| attendance | 0.2361 | 0.2361 |
| sport | 0.0951 | 0.0951 |
| marks_again | — | **0.4399** |

`marks` collapses from 0.5306 to 0.0906 and looks like the *least* important column in
the table — while 0.0906 + 0.4399 = **0.5306**, exactly its original value. The other
three columns do not move at all.

We added **no information whatsoever** — the new column is a literal copy — and the
apparent importance of marks fell by 83%.

So: **feature importance is a property of the fitted tree, not of the world.** Whenever
two columns carry the same information, the tree picks one essentially arbitrarily at
each split and the credit is divided between them. Real datasets are full of
near-duplicate columns (height and weight, revenue and headcount), so a low importance
score is never on its own evidence that a column does not matter.

Two further cautions: importances are biased **upward** for columns with many distinct
values (`family_income` gets 846 chances to be chosen, `sport` gets 1), and they describe
association, never cause.

### D3
Fit a depth-3 tree on the scholarship data and print it with `export_text`.

Now write the committee's rule **in plain English**, as you would put it in a letter to a
student. Then compare your version with the actual rule used to generate the data (it is
in the setup cell at the top). What did the tree get right, and what did it miss?

In [ ]:
# your code here

In [ ]:
t3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, y)
print(export_text(t3, feature_names=list(X.columns), decimals=0))

print("the paths that end in AWARDED:")
print("  1. marks <= 74, plays a sport, and marks > 62")
print("  2. marks > 74 and attendance <= 78, with marks > 86")
print("  3. marks > 74, attendance > 78, and income <= 704710")

**D3 answer**

A reasonable plain-English version of the depth-3 tree:

> You are awarded a scholarship if **either**
> (a) your marks are above 74 and your attendance is above 78, provided your family
> income is at or below about 7 lakh; **or**
> (b) your marks are above 86, even if your attendance is lower; **or**
> (c) you play a sport and your marks are above 62.

**What the tree got right.** Compare with the real rule in the setup cell:

| the real rule | the tree |
|---|---|
| marks ≥ 75 **and** attendance ≥ 80 | marks > 74, attendance > 78 ✔ |
| marks ≥ 88 | marks > 86 ✔ |
| sport **and** marks ≥ 62 **and** attendance ≥ 75 | sport and marks > 62 — attendance condition **missing** |
| **and** income ≤ 700000 throughout | income ≤ 704710, on one branch only |

All three award routes were recovered, and two of the thresholds are within about 2 of
the truth. From 900 noisy decisions and no hints, that is a real achievement — and you
can read it, which is the whole point of today.

**What it missed, and why.**

1. **The attendance condition on the sport route is gone.** Depth 3 ran out of room: that
   branch had already spent its three questions on marks, sport and marks again.
2. **The income cap appears on only one branch.** In the real rule it applies to
   everyone. The tree has no way to express "this condition applies everywhere" — it must
   re-discover the same cut separately in every branch where it matters, and with only
   50 students on the far side of it, most branches never find it worth asking.
3. **The thresholds are off by a little** (74.5 for 75, 86.5 for 88) because 8% of the
   decisions in the data contradict the rule, so the cleanest cut in the data is not
   exactly the cut in the rule.

Point 2 is the deepest one. A tree represents an AND-rule naturally and a
**shared condition awkwardly**, because each branch is built in isolation. Grow it
deeper and it recovers the missing pieces — at the cost of the readability that made it
worth reading in the first place. That tension never fully goes away.

---
### End of solutions

Every number above came from running the code in this notebook.